# Pseudoscalars from equivariant features

Enumerate the pseudoscalar (`0o`) paths reachable from a set of e3nn irreps, build a block that converts equivariant features into pseudoscalars, and verify they flip sign under reflection — the chirality signal.

Runs on CPU, no model needed. See [docs/pseudoscalars.md](../docs/pseudoscalars.md).

## 1. Which pseudoscalars are reachable?

`find_pseudoscalar_paths` is the universal enumerator: every valid 2-tensor-product path from the given irreps to `0o`.

In [ ]:
from e3nn.o3 import Irreps

from remedi.utils.pseudoscalar_paths import find_pseudoscalar_paths

irreps = Irreps("8x0e + 8x1o + 8x2e")  # e.g. MACE-style node irreps
paths = find_pseudoscalar_paths(irreps)
print(f"{len(paths)} path(s):")
for p in paths:
    print(" ", p)  # ir_in1 x ir_in2 -> ir_mid ; ir_mid x ir_in3 -> 0o

## 2. Convert features to pseudoscalars

`Rem3DiPseudoScalarTP` realizes all discovered paths as two fused tensor products, mapping `[..., irreps.dim]` features to `K` pseudoscalars.

In [ ]:
import torch

from remedi.model.preprocessing.pseudoscalar_tp import Rem3DiPseudoScalarTP

torch.manual_seed(0)
ps_tp = Rem3DiPseudoScalarTP(
    input_irreps=irreps,
    pseudoscalar_dimension=4,  # K
    invariant_conditioned=False,  # standalone block (no invariant-conditioned weights)
).eval()

x = torch.randn(16, irreps.dim)  # a batch of equivariant features
with torch.no_grad():
    ps = ps_tp(x)
print("pseudoscalars:", tuple(ps.shape))  # (16, 4)

## 3. Verify: pseudoscalars flip sign under reflection

Apply an improper transformation (a reflection, det = -1) to the input. A `0o` output must negate — this is exactly what makes it chirality-sensitive.

In [ ]:
R = torch.diag(torch.tensor([1.0, 1.0, -1.0]))  # mirror through the xy-plane
D = irreps.D_from_matrix(R)  # how the features transform
with torch.no_grad():
    ps_reflected = ps_tp(x @ D.T)

print("max|ps_reflected + ps| =", float((ps_reflected + ps).abs().max()))
assert torch.allclose(ps_reflected, -ps, atol=1e-4), "pseudoscalars should flip sign"
print("OK: reflection negates the pseudoscalars")